In [30]:
import time

# We will need Qibo to write our quantum circuit
from qibo import Circuit, gates, set_backend

# We will need mpstab to construct a surrogate
from mpstab import HSMPO

In [31]:
n = 10

circ = Circuit(nqubits=n)
circ.add(gates.H(0))
[circ.add(gates.CNOT(q, q+1)) for q in range(n - 1)]

circ.draw()

0: ─H─o─────────────────
1: ───X─o───────────────
2: ─────X─o─────────────
3: ───────X─o───────────
4: ─────────X─o─────────
5: ───────────X─o───────
6: ─────────────X─o─────
7: ───────────────X─o───
8: ─────────────────X─o─
9: ───────────────────X─


In [32]:
surry = HSMPO(circ)

In [33]:
%time
surry.expectation(
    observable="Z"*n
)

CPU times: user 2 μs, sys: 0 ns, total: 2 μs
Wall time: 2.15 μs


np.float64(1.0)

In [53]:
from mpstab.models.ansatze import HardwareEfficient 
from mpstab.utils import obs_string_to_qibo_hamiltonian

def execute_circuit(nqubits, nlayers):

    set_backend("numpy")
    obs_str = "Z" * nqubits

    ans = HardwareEfficient(nqubits=nqubits, nlayers=nlayers)
    hs = HSMPO(ansatz=ans)

    print(hs.expectation(obs_str))

    it = time.time()
    expval_mpstab = hs.expectation(obs_str)
    time_mpstab = time.time() - it
    
    ham = obs_string_to_qibo_hamiltonian(obs_str)
    it = time.time()
    expval_qibo = ham.expectation_from_state(
        ans.circuit().state()
    )
    time_qibo = time.time() - it
    return (expval_mpstab, expval_qibo), (time_mpstab, time_qibo)

In [59]:
execute_circuit(nqubits=7, nlayers=3)

[Qibo 0.2.23|INFO|2026-02-12 16:45:12]: Using numpy backend on /CPU:0


1.0


((np.float64(1.0), np.float64(0.060655740792993205)),
 (0.16927576065063477, 0.003439188003540039))